# E791 $D^+\to\pi^-\pi^+\pi^+$ — grid vs uniform-MC normalization closure

This notebook compares the two supported normalization methods on exactly the same pseudo-data and exactly the same randomized starting point. Absolute NLL values are not compared across methods; all likelihood comparisons use truth-referenced $\Delta$NLL within each method.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DalitzMC, DalitzGrid, DecayChannel, DecayModel, Minimizer, NonResonant,
    Parameter, RealImag, Resonance, enable_x64, weighted_resample,
)
enable_x64()


In [ ]:
channel=DecayChannel("D+",("pi-","pi+","pi+"))
fit2_polar={"sigma":(1.17,205.7),"rho770":(1.0,0.0),"NR":(0.48,57.3),"f0_980":(0.43,165.0),"f2_1270":(0.76,57.3),"f0_1370":(0.26,105.4),"rho1450":(0.14,319.1)}
def polar_to_xy(r,p):
    p=np.deg2rad(p); return r*np.cos(p),r*np.sin(p)
def internal_xy(name):
    r,p=fit2_polar[name]
    if name=="NR": p+=180.0
    return polar_to_xy(r,p)
truth_xy={n:internal_xy(n) for n in fit2_polar}
truth={}
def free_coeff(name):
    x,y=truth_xy[name]; truth[f"{name}.x"]=float(x); truth[f"{name}.y"]=float(y)
    return RealImag(Parameter.coefficient(f"{name}.x",0.0,owner=name,step=0.01),Parameter.coefficient(f"{name}.y",0.0,owner=name,step=0.01))
c={"sigma":free_coeff("sigma"),"rho770":RealImag(1.0,0.0),"NR":free_coeff("NR"),"f0_980":free_coeff("f0_980"),"f2_1270":free_coeff("f2_1270"),"f0_1370":free_coeff("f0_1370"),"rho1450":free_coeff("rho1450")}
model=DecayModel(channel,[
    Resonance("sigma",(0,1),c["sigma"],mass=0.4780,width=0.3240,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("rho770",(0,1),c["rho770"],mass=0.7693,width=0.1502,spin=1,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f0_980",(0,1),c["f0_980"],mass=0.9750,width=0.0440,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f2_1270",(0,1),c["f2_1270"],mass=1.2750,width=0.1850,spin=2,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f0_1370",(0,1),c["f0_1370"],mass=1.4340,width=0.1730,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("rho1450",(0,1),c["rho1450"],mass=1.4650,width=0.3100,spin=1,resonance_radius=3.0,parent_radius=3.0),
    NonResonant(c["NR"]),
])


## 1. Build both normalization samples


In [ ]:
N_NORM=1_000_000
GRID_N=1000
norm_mc=DalitzMC(channel.parent_mass,channel.daughter_masses).generate(N_NORM,seed=2027)
norm_grid=DalitzGrid(channel.parent_mass,channel.daughter_masses,resolution=GRID_N).sample()
print("MC points   =",norm_mc.size)
print("grid points =",norm_grid.size)
print("MC constant weights =",bool(jnp.all(norm_mc.weights==norm_mc.weights[0])))


## 2. Generate one common pseudo-data sample


In [ ]:
N_POOL=1_000_000; N_DATA=100_000
pool=model.generate_phase_space(N_POOL,seed=2000)
truth_cache_pool=model.prepare_cache(pool,norm_grid)
truth_intensity,_=truth_cache_pool.evaluate(truth)
data=weighted_resample(jax.random.key(791),pool,pool.weights*truth_intensity,N_DATA,replace=True)
print("toy events =",data.size)


## 3. Define MC and grid likelihoods


In [ ]:
cache_mc=model.prepare_cache(data,norm_mc)
cache_grid=model.prepare_cache(data,norm_grid)
def nll_from_cache(cache,values):
    intensity,normalization=cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity,min=1e-300)))+data.size*jnp.log(normalization)
def nll_mc(values): return nll_from_cache(cache_mc,values)
def nll_grid(values): return nll_from_cache(cache_grid,values)
nll_mc_truth=float(nll_mc(truth)); nll_grid_truth=float(nll_grid(truth))
def delta_nll_mc(values): return float(nll_mc(values))-nll_mc_truth
def delta_nll_grid(values): return float(nll_grid(values))-nll_grid_truth


## 4. Normalization-matrix comparison


In [ ]:
M_mc=np.asarray(cache_mc.normalization_matrix_fixed); M_grid=np.asarray(cache_grid.normalization_matrix_fixed)
delta=M_mc-M_grid; mask=np.abs(M_grid)>1e-6
print("max |M_MC-M_grid| =",float(np.max(np.abs(delta))))
print("RMS |M_MC-M_grid| =",float(np.sqrt(np.mean(np.abs(delta)**2))))
print("max relative difference (|M_grid|>1e-6) =",float(np.max(np.abs(delta[mask])/np.abs(M_grid[mask]))))
print("MC eigenvalues   =",np.linalg.eigvalsh(M_mc))
print("grid eigenvalues =",np.linalg.eigvalsh(M_grid))


## 5. One common randomized start


In [ ]:
rng=np.random.default_rng(314159)
start_values={p.name:float(rng.uniform(-2.5,2.5)) for p in model.parameters if not p.fixed}
print("DeltaNLL_MC(start)   =",delta_nll_mc(start_values))
print("DeltaNLL_grid(start) =",delta_nll_grid(start_values))


## 6. Truth-referenced NLL path from start to truth


In [ ]:
free_names=[p.name for p in model.parameters if not p.fixed]
ts=np.linspace(0,1,81); mc_scan=[]; grid_scan=[]
for t in ts:
    point={n:(1-t)*start_values[n]+t*truth[n] for n in free_names}
    mc_scan.append(delta_nll_mc(point)); grid_scan.append(delta_nll_grid(point))
fig,ax=plt.subplots(figsize=(8,5))
ax.plot(ts,mc_scan,label="MC"); ax.plot(ts,grid_scan,linestyle="--",label="grid")
ax.axhline(0.0,linewidth=1.0); ax.set_xlabel("t: start -> truth"); ax.set_ylabel(r"$\Delta$NLL relative to truth"); ax.legend(); plt.show()


## 7. Fit the same toy with MC and grid normalization


In [ ]:
min_mc=Minimizer(nll_mc,model.parameters,tolerance=1e-4,verbose=2)
min_grid=Minimizer(nll_grid,model.parameters,tolerance=1e-4,verbose=2)
print("MC gradient check")
check_mc=min_mc.check_gradient(start_values,step_scale=1e-5,print_table=True)
print("grid gradient check")
check_grid=min_grid.check_gradient(start_values,step_scale=1e-5,print_table=True)
result_mc=min_mc.fit(start_values=start_values,simplex=False,ncall=100000)
result_grid=min_grid.fit(start_values=start_values,simplex=False,ncall=100000)
fit_mc={p.name:float(result_mc.values[p.name]) for p in model.parameters if not p.fixed}
fit_grid={p.name:float(result_grid.values[p.name]) for p in model.parameters if not p.fixed}
print("MC   valid / DeltaNLL / EDM =",bool(result_mc.valid),delta_nll_mc(fit_mc),float(result_mc.fmin.edm))
print("grid valid / DeltaNLL / EDM =",bool(result_grid.valid),delta_nll_grid(fit_grid),float(result_grid.fmin.edm))


## 8. Parameter comparison: truth, MC fit and grid fit


In [ ]:
names=[p.name for p in model.parameters if not p.fixed]
x=np.arange(len(names))
truth_arr=np.array([truth[n] for n in names])
mc_arr=np.array([fit_mc[n] for n in names]); grid_arr=np.array([fit_grid[n] for n in names])
mc_err=np.array([result_mc.errors[n] for n in names],dtype=float); grid_err=np.array([result_grid.errors[n] for n in names],dtype=float)
fig,ax=plt.subplots(figsize=(13,6))
ax.scatter(x,truth_arr,marker="x",s=70,label="truth")
ax.errorbar(x-0.10,mc_arr,yerr=mc_err,fmt=".",capsize=3,label="MC fit")
ax.errorbar(x+0.10,grid_arr,yerr=grid_err,fmt=".",capsize=3,label="grid fit")
ax.set_xticks(x); ax.set_xticklabels(names,rotation=55,ha="right")
ax.set_ylabel("Cartesian coefficient"); ax.set_title("Coefficient closure: MC vs grid normalization"); ax.legend(); plt.tight_layout(); plt.show()


## 9. Pull comparison


In [ ]:
pull_mc=(mc_arr-truth_arr)/mc_err; pull_grid=(grid_arr-truth_arr)/grid_err
fig,ax=plt.subplots(figsize=(13,5))
w=0.38
ax.bar(x-w/2,pull_mc,width=w,label="MC fit")
ax.bar(x+w/2,pull_grid,width=w,label="grid fit")
ax.axhline(0.0,linewidth=1.0); ax.axhline(1.0,linestyle="--",linewidth=1.0); ax.axhline(-1.0,linestyle="--",linewidth=1.0)
ax.set_xticks(x); ax.set_xticklabels(names,rotation=55,ha="right")
ax.set_ylabel("pull"); ax.set_title("Pull comparison: MC vs grid normalization"); ax.legend(); plt.tight_layout(); plt.show()


## 10. Projection comparison: toy, truth, MC fit and grid fit


In [ ]:
proj_cache_grid=model.prepare_cache(pool,norm_grid)
def projection(values,bins):
    intensity,_=proj_cache_grid.evaluate(values)
    w=np.asarray(pool.weights*intensity)
    h12,_=np.histogram(np.asarray(pool.s12),bins=bins,weights=w)
    h13,_=np.histogram(np.asarray(pool.s13),bins=bins,weights=w)
    return h12+h13
sdata=np.concatenate([np.asarray(data.s12),np.asarray(data.s13)])
bins=np.linspace(sdata.min(),sdata.max(),110); centers=0.5*(bins[:-1]+bins[1:])
hd,_=np.histogram(sdata,bins=bins)
ht=projection(truth,bins); hmc=projection(fit_mc,bins); hgrid=projection(fit_grid,bins)
for h in (ht,hmc,hgrid): h*=hd.sum()/h.sum()
fig,ax=plt.subplots(figsize=(10,5.5))
ax.errorbar(centers,hd,yerr=np.sqrt(np.maximum(hd,1)),fmt=".",label="toy")
ax.step(centers,ht,where="mid",linestyle="--",label="truth")
ax.step(centers,hmc,where="mid",label="MC fit")
ax.step(centers,hgrid,where="mid",label="grid fit")
ax.set_xlabel(r"$m^2(\pi^-\pi^+)$ [GeV$^2$]"); ax.set_ylabel("events / bin")
ax.set_title("Projection comparison: MC vs grid normalization"); ax.legend(); plt.tight_layout(); plt.show()


## 11. Cross-evaluate both fitted points


In [ ]:
print("DeltaNLL_MC(MC fit)     =",delta_nll_mc(fit_mc))
print("DeltaNLL_MC(grid fit)   =",delta_nll_mc(fit_grid))
print("DeltaNLL_grid(MC fit)   =",delta_nll_grid(fit_mc))
print("DeltaNLL_grid(grid fit) =",delta_nll_grid(fit_grid))
